In [ ]:
import sys
actual_path = "/data2/home/luodh/Git-workflow/Workflow_for_vasp_catalysis/vasp_catalysis_tools"

if actual_path not in sys.path:
    sys.path.insert(0, actual_path)

import pymatgen


In [2]:
API_KEY = "n7GbikHqL3SlQCH3diuPBBlMNqRcTnFR"
import logging
logging.basicConfig(level=logging.INFO)

from core.search import MPQueryService, detect_query_mode

# 初始化服务（only_stable=True 只返回稳定结构）
svc = MPQueryService(
    api_key=API_KEY,
    only_stable=True,
    max_results=5,       # 测试时限制数量，避免等待过久
    cache_ttl=300,
)
print("✅ MPQueryService 初始化成功")

✅ MPQueryService 初始化成功


In [3]:
test_inputs = [
    "Pt",           # → formula
    "Fe2O3",        # → formula
    "LiFePO4",      # → formula
    "Fe-O",         # → chemsys
    "Li-Fe-P-O",    # → chemsys
    "Fe, O",        # → elements
    "Fe O",         # → elements
    "mp-126",       # → material_id
]

print(f"{'输入':<15} {'识别模式':<15} {'查询参数'}")
print("-" * 55)
for inp in test_inputs:
    detected = detect_query_mode(inp)
    mode = detected.pop("_mode")
    print(f"{inp:<15} {mode:<15} {detected}")

输入              识别模式            查询参数
-------------------------------------------------------
Pt              formula         {'formula': 'Pt'}
Fe2O3           formula         {'formula': 'Fe2O3'}
LiFePO4         formula         {'formula': 'LiFePO4'}
Fe-O            chemsys         {'chemsys': 'Fe-O'}
Li-Fe-P-O       chemsys         {'chemsys': 'Li-Fe-P-O'}
Fe, O           elements        {'elements': ['Fe', 'O']}
Fe O            elements        {'elements': ['Fe', 'O']}
mp-126          material_id     {'material_ids': ['mp-126']}


In [4]:
# ════════════════════════════════════════════
# Cell 5 ▶ 按化学式查询（formula）
# ════════════════════════════════════════════

results = svc.query("Pt")

print(f"\n查询 'Pt' → {len(results)} 条结果\n")
for r in results:
    print(f"  {r['material_id']:<12} {r['formula']:<10} "
          f"SG: {r['space_group']:<8} {r['crystal_system']:<10} "
          f"a={r['a']} b={r['b']} c={r['c']} Å")

2026-04-06 22:03:39,761 [INFO] [query] mode=formula  input='Pt'
INFO:MPQueryService:[query] mode=formula  input='Pt'


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]


查询 'Pt' → 1 条结果

  mp-126       Pt         SG: Fm-3m    Cubic      a=3.9432 b=3.9432 c=3.9432 Å


In [5]:
# ════════════════════════════════════════════
# Cell 6 ▶ 按化学体系查询（chemsys）
# ════════════════════════════════════════════

results = svc.query("Fe-O")

print(f"\n查询 'Fe-O' → {len(results)} 条结果\n")
for r in results:
    print(f"  {r['material_id']:<12} {r['formula']:<10} "
          f"SG: {r['space_group']:<8} {r['crystal_system']}")

2026-04-06 22:04:30,762 [INFO] [query] mode=chemsys  input='Fe-O'
INFO:MPQueryService:[query] mode=chemsys  input='Fe-O'


Retrieving SummaryDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]


查询 'Fe-O' → 2 条结果

  mp-19770     Fe2O3      SG: R-3c     Trigonal
  mp-1274279   FeO        SG: C2/m     Monoclinic


In [6]:
# ════════════════════════════════════════════
# Cell 7 ▶ 按 material_id 查询
# ════════════════════════════════════════════

results = svc.query("mp-126")

if results:
    r = results[0]
    print(f"\n查询 mp-126：")
    print(f"  化学式     : {r['formula']}")
    print(f"  空间群     : {r['space_group']}  ({r['crystal_system']})")
    print(f"  晶格参数   : a={r['a']} b={r['b']} c={r['c']} Å")
    print(f"  α β γ      : {r['alpha']} {r['beta']} {r['gamma']} °")

2026-04-06 22:05:09,554 [INFO] [query] mode=material_id  input='mp-126'
INFO:MPQueryService:[query] mode=material_id  input='mp-126'


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]


查询 mp-126：
  化学式     : Pt
  空间群     : Fm-3m  (Cubic)
  晶格参数   : a=3.9432 b=3.9432 c=3.9432 Å
  α β γ      : 90.0 90.0 90.0 °


In [7]:
# ════════════════════════════════════════════
# Cell 8 ▶ 查看 XYZ 字符串（3Dmol 渲染用）
# ════════════════════════════════════════════

results = svc.query("mp-126")

if results:
    xyz = results[0]["xyz"]
    print("── XYZ 字符串（前5行）──")
    for line in xyz.split("\n")[:5]:
        print(" ", line)
    print(f"  ... 共 {len(xyz.splitlines())} 行")

2026-04-06 22:05:34,671 [INFO] [cache] material_id::mp-126
INFO:MPQueryService:[cache] material_id::mp-126


── XYZ 字符串（前5行）──
  4
  Pt | mp-126
  Pt 0.000000 0.000000 0.000000
  Pt 0.000000 1.971575 1.971575
  Pt 1.971575 0.000000 1.971575
  ... 共 6 行


In [8]:
# ════════════════════════════════════════════
# Cell 8 ▶ 查看 XYZ 字符串（3Dmol 渲染用）
# ════════════════════════════════════════════

results = svc.query("mp-126")

if results:
    xyz = results[0]["xyz"]
    print("── XYZ 字符串（前5行）──")
    for line in xyz.split("\n")[:5]:
        print(" ", line)
    print(f"  ... 共 {len(xyz.splitlines())} 行")

# ════════════════════════════════════════════
# Cell 9 ▶ 查看 CIF 字符串
# ════════════════════════════════════════════

results = svc.query("mp-126")

if results:
    cif = results[0]["cif"]
    print("── CIF 字符串（前10行）──")
    for line in cif.split("\n")[:10]:
        print(" ", line)

# ════════════════════════════════════════════
# Cell 10 ▶ 查看 POSCAR 字符串
# ════════════════════════════════════════════

results = svc.query("mp-126")

if results:
    poscar = results[0]["poscar"]
    print("── POSCAR 字符串 ──")
    print(poscar)

# ════════════════════════════════════════════
# Cell 11 ▶ 获取 pymatgen Structure 对象（后端计算用）
# ════════════════════════════════════════════

struct = svc.get_structure("mp-126")

if struct:
    print(f"pymatgen Structure 对象：")
    print(f"  类型       : {type(struct)}")
    print(f"  化学式     : {struct.composition.reduced_formula}")
    print(f"  原子数     : {len(struct)}")
    print(f"  晶格       : {struct.lattice}")
    print(f"\n  可直接用于：")
    print(f"    - SpacegroupAnalyzer(struct)   # 对称性分析")
    print(f"    - struct.get_primitive_structure()  # 原胞")
    print(f"    - AseAtomsAdaptor.get_atoms(struct) # 转 ASE")


2026-04-06 22:05:59,588 [INFO] [cache] material_id::mp-126
INFO:MPQueryService:[cache] material_id::mp-126
2026-04-06 22:05:59,591 [INFO] [cache] material_id::mp-126
INFO:MPQueryService:[cache] material_id::mp-126
2026-04-06 22:05:59,593 [INFO] [cache] material_id::mp-126
INFO:MPQueryService:[cache] material_id::mp-126
2026-04-06 22:05:59,595 [INFO] [cache] material_id::mp-126
INFO:MPQueryService:[cache] material_id::mp-126


── XYZ 字符串（前5行）──
  4
  Pt | mp-126
  Pt 0.000000 0.000000 0.000000
  Pt 0.000000 1.971575 1.971575
  Pt 1.971575 0.000000 1.971575
  ... 共 6 行
── CIF 字符串（前10行）──
  # generated using pymatgen
  data_Pt
  _symmetry_space_group_name_H-M   'P 1'
  _cell_length_a   3.94315036
  _cell_length_b   3.94315036
  _cell_length_c   3.94315036
  _cell_angle_alpha   90.00000000
  _cell_angle_beta   90.00000000
  _cell_angle_gamma   90.00000000
  _symmetry_Int_Tables_number   1
── POSCAR 字符串 ──
Pt | mp-126
1.0
   3.9431503618848787    0.0000000000000000    0.0000000000000002
   0.0000000000000006    3.9431503618848787    0.0000000000000002
   0.0000000000000000    0.0000000000000000    3.9431503618848787
Pt
4
direct
   0.0000000000000000    0.0000000000000000    0.0000000000000000 Pt
   0.0000000000000000    0.5000000000000000    0.5000000000000000 Pt
   0.5000000000000000    0.0000000000000000    0.5000000000000000 Pt
   0.5000000000000000    0.5000000000000000    0.0000000000000000 Pt

pymatgen Str

In [9]:
# ════════════════════════════════════════════
# Cell 12 ▶ 保存结构到本地磁盘（xyz / cif / poscar）
# ════════════════════════════════════════════

saved_paths = svc.save_to_disk(
    material_id="mp-126",
    save_dir="./test_structures",   # 保存目录（自动创建）
    fmt="all",                       # "xyz" | "cif" | "poscar" | "all"
)

print("\n已保存文件：")
for fmt, path in saved_paths.items():
    print(f"  {fmt.upper():<8} → {path}")

# ════════════════════════════════════════════
# Cell 13 ▶ 验证保存的文件内容
# ════════════════════════════════════════════

from pathlib import Path

for fmt, path in saved_paths.items():
    p = Path(path)
    lines = p.read_text(encoding="utf-8").splitlines()
    print(f"\n── {fmt.upper()} ({p.name})，共 {len(lines)} 行，前3行：")
    for line in lines[:3]:
        print(f"   {line}")

# ════════════════════════════════════════════
# Cell 14 ▶ 缓存测试（第二次查询应命中缓存）
# ════════════════════════════════════════════

import time

print("第一次查询（请求 API）：")
t0 = time.time()
svc.clear_cache()
svc.query("Pt")
print(f"  耗时 {time.time()-t0:.2f}s")

print("\n第二次查询（命中缓存）：")
t0 = time.time()
svc.query("Pt")
print(f"  耗时 {time.time()-t0:.4f}s  ← 应远快于第一次")

# ════════════════════════════════════════════
# Cell 15 ▶ 错误处理测试
# ════════════════════════════════════════════

# 测试空输入
try:
    svc.query("")
except ValueError as e:
    print(f"空输入错误（预期）：{e}")

# 测试不存在的 material_id
results = svc.query("mp-9999999999")
print(f"不存在的 ID 返回：{results}（预期空列表）")

# 测试无稳定结构的化学式（返回全部结果并警告）
results = svc.query("Og")   # 第118号元素，极少稳定结构
print(f"查询 'Og' 返回 {len(results)} 条结果")

2026-04-06 22:06:34,355 [INFO] [cache] material_id::mp-126
INFO:MPQueryService:[cache] material_id::mp-126
2026-04-06 22:06:34,360 [INFO] 已保存 XYZ → test_structures/mp-126_Pt.xyz
INFO:MPQueryService:已保存 XYZ → test_structures/mp-126_Pt.xyz
2026-04-06 22:06:34,366 [INFO] 已保存 CIF → test_structures/mp-126_Pt.cif
INFO:MPQueryService:已保存 CIF → test_structures/mp-126_Pt.cif
2026-04-06 22:06:34,370 [INFO] 已保存 POSCAR → test_structures/mp-126_Pt.POSCAR
INFO:MPQueryService:已保存 POSCAR → test_structures/mp-126_Pt.POSCAR
2026-04-06 22:06:34,375 [INFO] 缓存已清空
INFO:MPQueryService:缓存已清空
2026-04-06 22:06:34,376 [INFO] [query] mode=formula  input='Pt'
INFO:MPQueryService:[query] mode=formula  input='Pt'



已保存文件：
  XYZ      → test_structures/mp-126_Pt.xyz
  CIF      → test_structures/mp-126_Pt.cif
  POSCAR   → test_structures/mp-126_Pt.POSCAR

── XYZ (mp-126_Pt.xyz)，共 6 行，前3行：
   4
   mp-126_Pt
   Pt 0.000000 0.000000 0.000000

── CIF (mp-126_Pt.cif)，共 30 行，前3行：
   # generated using pymatgen
   data_Pt
   _symmetry_space_group_name_H-M   'P 1'

── POSCAR (mp-126_Pt.POSCAR)，共 12 行，前3行：
   mp-126_Pt
   1.0
      3.9431503618848787    0.0000000000000000    0.0000000000000002
第一次查询（请求 API）：


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-06 22:06:35,771 [INFO] [cache] formula::Pt
INFO:MPQueryService:[cache] formula::Pt
2026-04-06 22:06:35,773 [INFO] [query] mode=material_id  input='mp-9999999999'
INFO:MPQueryService:[query] mode=material_id  input='mp-9999999999'


  耗时 1.40s

第二次查询（命中缓存）：
  耗时 0.0018s  ← 应远快于第一次
空输入错误（预期）：查询内容不能为空


...

2026-04-06 22:06:37,665 [INFO] [query] 无结果：'mp-9999999999'
INFO:MPQueryService:[query] 无结果：'mp-9999999999'
2026-04-06 22:06:37,668 [INFO] [query] mode=formula  input='Og'
INFO:MPQueryService:[query] mode=formula  input='Og'


不存在的 ID 返回：[]（预期空列表）


Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

2026-04-06 22:06:38,247 [INFO] [query] 无结果：'Og'
INFO:MPQueryService:[query] 无结果：'Og'


查询 'Og' 返回 0 条结果
